# 📬 Customer Feedback Analysis & Automated Response

**Goal:** Find the urgent, negative customer reviews using simple Python rules (no Machine Learning), discover the most common complaint keywords, and use Generative AI to draft empathetic apology emails for the 3 most critical reviews.

In [ ]:
import re
import os
import time
import getpass
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from IPython.display import display, Markdown

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

> ⚠️ **Before running:**
> 1. Download **Women's E-Commerce Clothing Reviews** from Kaggle and update the file path in Step 1 below.
> 2. Install the Gen AI library once: `pip install google-genai`
> 3. Get a free Gemini API key from Google AI Studio. Step 13 will ask you to paste it (or set the `GEMINI_API_KEY` environment variable). **Never type your key directly inside the notebook.**

## Step 1: Load the Dataset

In [ ]:
# =================================================================
# STEP 1: READ THE DATA
# =================================================================
# WHY: We load the reviews CSV file from your folder.
# This dataset has the written review and a 1-5 star rating,
# which is exactly what we need to find unhappy customers.

df = pd.read_csv(r"Womens Clothing E-Commerce Reviews.csv")

# The first column is only a row number ("Unnamed: 0"), so we drop it.
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

rows_loaded = len(df)

print("STEP 1 - DATASET LOADED")
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(list(df.columns))
print("\nFirst 3 rows:")
print(df.head(3))

# INSIGHT: Each row is one customer review. The columns we care about
# most are "Review Text" (what the customer wrote) and "Rating" (1-5 stars).

## Step 2: Check for Missing Values

**WHY** : Missing information can give us wrong results. So first we check whether anything is empty.

In [ ]:
print("\nSTEP 2 - MISSING VALUES")
print(df.isna().sum().sort_values(ascending=False))

# INSIGHT: "Title" and "Review Text" are the columns with the most empty cells.
# A review with no text cannot be analysed, so we must handle these.

## Step 3: Fix Missing Values

In [ ]:
# =================================================================
# STEP 3: FIX MISSING VALUES
# =================================================================
# WHY: A review with no text (or no rating) tells us nothing,
# so we remove those rows. For small side columns like "Title",
# we fill the empty cells with a simple label so we do not lose
# the whole review just because of a missing title.

# 1. Remove rows where the important columns are empty
df = df.dropna(subset=["Review Text", "Rating"]).copy()

# 2. Remove reviews that contain only spaces
df = df[df["Review Text"].str.strip() != ""].copy()

# 3. Fill the less important columns with a label
df["Title"] = df["Title"].fillna("No Title")
for col in ["Division Name", "Department Name", "Class Name"]:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

# 4. Make sure Rating is a whole number (1, 2, 3, 4, 5)
df["Rating"] = df["Rating"].astype(int)

rows_after_missing = len(df)

print("\nSTEP 3 - MISSING VALUES FIXED")
print("Rows before:", rows_loaded)
print("Rows after :", rows_after_missing)
print("Rows removed:", rows_loaded - rows_after_missing)
print("\nMissing values left:")
print(df.isna().sum().sum())

# INSIGHT: We only removed rows that had NO review text.
# Everything else was filled, so we keep as much data as possible.

## Step 4: Check and Remove Duplicate Reviews

In [ ]:
# =================================================================
# STEP 4: REMOVE REPEATED REVIEWS
# =================================================================
# WHY: If the same review is saved twice, its complaint words
# would be counted twice and make some problems look bigger
# than they really are.

duplicate_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

print("\nSTEP 4 - DUPLICATES")
print("Duplicate rows found and removed:", duplicate_count)
print("Rows now:", len(df))

# INSIGHT: If the number above is 0, the file had no exact repeats.
# Either way, our data is now one-row-per-review.

## Step 5: Clean the Review Text

In [ ]:
# =================================================================
# STEP 5: CLEAN THE TEXT
# =================================================================
# WHY: Python sees "Broken", "broken" and "broken!!!" as three
# different words. We make them all look the same:
#   1. lowercase everything
#   2. remove apostrophes (don't -> dont)
#   3. remove numbers, symbols and punctuation
#   4. remove extra spaces
# We keep the ORIGINAL "Review Text" untouched, because the AI
# should read the customer's real words in Step 14.

def clean_text(text):
    text = str(text).lower()                    # 1. lowercase
    text = text.replace("'", "").replace("’", "")   # 2. don't -> dont
    text = re.sub(r"[^a-z\s]", " ", text)       # 3. keep only letters and spaces
    text = re.sub(r"\s+", " ", text).strip()    # 4. remove extra spaces
    return text

df["clean_text"] = df["Review Text"].apply(clean_text)

# Count words in each review (we use this later to find "detailed" reviews)
df["word_count"] = df["clean_text"].str.split().str.len()

print("\nSTEP 5 - TEXT CLEANED")
for i in range(3):
    print(f"\nBEFORE: {df['Review Text'].iloc[i]}")
    print(f"AFTER : {df['clean_text'].iloc[i]}")

# INSIGHT: The cleaned text has no capital letters, symbols or numbers,
# so counting words is now fair and simple.

## Step 6: Look at the Rating Distribution

### 💡 Insight
The rating distribution gives the first overall picture of customer satisfaction. We look at the full dataset first, then focus on critical reviews.


In [ ]:
# =================================================================
# STEP 6: HOW ARE THE RATINGS SPREAD?
# =================================================================
# WHY: Before filtering, we want to know how many unhappy
# customers there are compared to happy ones.

rating_counts = df["Rating"].value_counts().sort_index()

print("\nSTEP 6 - RATING COUNTS")
print(rating_counts)

# Red bars = ratings 1 and 2 (the ones we will call "Critical")
bar_colors = ["#e74c3c" if r <= 2 else "#3498db" for r in rating_counts.index]
rating_counts.plot(kind="bar", color=bar_colors, rot=0)
plt.title("Number of Reviews by Star Rating (Red = Critical)", fontsize=14)
plt.xlabel("Star Rating")
plt.ylabel("Number of Reviews")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

# INSIGHT: Most reviews are positive (4-5 stars). The unhappy
# customers (red bars) are a small group hiding inside a big file,
# which is why manual reading is slow and filtering helps.

## Step 7: Filter the Critical Reviews (Rule-Based, No ML)

In [ ]:
# =================================================================
# STEP 7: FILTER CRITICAL REVIEWS
# =================================================================
# WHY: Our rule is simple and easy to explain to a manager:
#   Rating 1 or 2  ->  "Critical"
# No Machine Learning is used, just a normal pandas filter.

CRITICAL_RATINGS = [1, 2]

df_critical = df[df["Rating"].isin(CRITICAL_RATINGS)].copy()
critical_percent = round(len(df_critical) / len(df) * 100, 1)

print("\nSTEP 7 - CRITICAL REVIEWS")
print("Total reviews   :", len(df))
print("Critical reviews:", len(df_critical))
print(f"Share of all reviews: {critical_percent}%")
print("\nBreakdown:")
print(df_critical["Rating"].value_counts().sort_index())

# INSIGHT: The support team now only has to look at this smaller
# table instead of every review.

## Step 8: Most Common Words in Critical Reviews

In [ ]:
# =================================================================
# STEP 8: MOST COMMON WORDS (WITH A STOP-WORD LIST)
# =================================================================
# WHY: Words like "the", "and", "is" appear everywhere and tell us
# nothing. We keep a list of such "stop words" and skip them.
# Then collections.Counter does the counting for us.

STOPWORDS = set("""
a about above after again all also am an and any are as at be because been before being below between
both but by can could did do does doing dont doesnt didnt down during each few for from further get got
had has have having he her here hers him his how i id if im in into is isnt it its ive just like me more
most my myself no nor not now of off on once only or other our out over own really same she should so
some such than that the their them then there these they this those through to too under until up very
was wasnt we were what when where which while who why will with would you your yours
one even much still back way think thought bought ordered order wear wore worn look looks looked
dress top shirt sweater pants skirt jeans blouse
""".split())

def get_top_words(texts, top_n=15):
    """Return the top_n most common words from a list of cleaned reviews."""
    counter = Counter()
    for text in texts:
        for word in text.split():
            if word not in STOPWORDS and len(word) > 2:
                counter[word] += 1
    return counter.most_common(top_n)

critical_texts = df_critical["clean_text"].tolist()

top_words = get_top_words(critical_texts, top_n=15)
top_words_df = pd.DataFrame(top_words, columns=["Word", "Count"])

print("\nSTEP 8 - TOP 15 WORDS IN CRITICAL REVIEWS")
print(top_words_df)

top_words_df.sort_values("Count").plot(
    kind="barh", x="Word", y="Count", color="#e74c3c", legend=False
)
plt.title("Most Common Words in Critical Reviews", fontsize=14)
plt.xlabel("Times Used")
plt.ylabel("Word")
plt.tight_layout()
plt.show()

# INSIGHT: This shows the general vocabulary of unhappy customers.
# Words about size, fit, fabric and quality usually appear near the top,
# but some words (like "love") are not complaints by themselves.
# So in Step 9 we also check a focused complaint list.

## Step 9: Complaint Keyword Finder

### 💡 Insight
Complaint keywords convert free-text customer comments into measurable problems. This makes it easier to see what customers are unhappy about instead of reading every review manually.


In [ ]:
# =================================================================
# STEP 9: COMPLAINT KEYWORD WATCH-LIST
# =================================================================
# WHY: The top words in Step 8 include some neutral words.
# Here we make our own list of words (and short phrases) that
# clearly mean a problem, grouped by type of complaint.
# We then count in how many critical reviews each one appears.
# (You can add your own words, e.g. "rude", "late", "broken".)

complaint_groups = {
    "Fit & Size": [
        "too small", "too big", "too tight", "too large", "too loose",
        "too short", "too long", "runs small", "runs large",
        "baggy", "boxy", "tent"
    ],
    "Quality & Fabric": [
        "cheap", "poor quality", "poorly made", "thin", "flimsy", "rough",
        "shrunk", "faded", "pilled", "ripped", "torn", "stitching",
        "seams", "hole", "holes", "wrinkled", "fell apart"
    ],
    "Looks & Color": [
        "see through", "sheer", "ugly", "tacky", "frumpy",
        "unflattering", "not as pictured", "misleading"
    ],
    "Comfort": [
        "uncomfortable", "itchy", "scratchy", "stiff", "heavy", "hot"
    ],
    "Service & Delivery": [
        "late", "delay", "delayed", "damaged", "broken", "wrong",
        "missing", "rude", "refund", "exchange", "customer service"
    ],
    "Disappointment & Returns": [
        "disappointed", "disappointing", "disappointment", "terrible",
        "awful", "horrible", "worst", "waste", "regret",
        "return", "returned", "returning"
    ],
}

# One flat list of every keyword, and a map from keyword -> its group
all_keywords = [kw for words in complaint_groups.values() for kw in words]
keyword_to_group = {kw: group for group, words in complaint_groups.items() for kw in words}

def count_complaint_keywords(texts, keyword_list):
    """Count in how many reviews each keyword appears (once per review)."""
    counter = Counter()
    for text in texts:
        padded = f" {text} "              # spaces around the text so we match WHOLE words only
        for kw in keyword_list:
            if f" {kw} " in padded:       # works for single words AND short phrases
                counter[kw] += 1
    return counter

keyword_counts = count_complaint_keywords(critical_texts, all_keywords)

keyword_df = pd.DataFrame(keyword_counts.most_common(), columns=["Keyword", "Reviews"])
keyword_df["Category"] = keyword_df["Keyword"].map(keyword_to_group)

print("\nSTEP 9 - TOP COMPLAINT KEYWORDS")
print(keyword_df.head(15))

keyword_df.head(15).sort_values("Reviews").plot(
    kind="barh", x="Keyword", y="Reviews", color="#e67e22", legend=False
)
plt.title("Top Complaint Keywords in Critical Reviews", fontsize=14)
plt.xlabel("Number of Critical Reviews Mentioning It")
plt.ylabel("Keyword")
plt.tight_layout()
plt.show()

# INSIGHT: These are real complaint words, not just common words.
# Keywords with a count of 0 are not shown, which means customers
# did not use them in this dataset.

## Step 10: Which Type of Complaint Is Biggest?

In [ ]:
# =================================================================
# STEP 10: COMPLAINT CATEGORIES
# =================================================================
# WHY: Single words are useful, but a manager wants the big picture:
# "Is our main problem sizing, fabric quality or delivery?"
# For each group we count the critical reviews that mention
# at least ONE keyword from that group.

def count_reviews_per_group(texts, groups):
    """For each group, count reviews that contain at least one of its keywords."""
    result = {}
    for group, words in groups.items():
        count = 0
        for text in texts:
            padded = f" {text} "
            if any(f" {w} " in padded for w in words):
                count += 1
        result[group] = count
    return result

group_counts = count_reviews_per_group(critical_texts, complaint_groups)

group_df = pd.DataFrame(list(group_counts.items()), columns=["Category", "Reviews"])
group_df["% of Critical Reviews"] = (group_df["Reviews"] / len(df_critical) * 100).round(1)
group_df = group_df.sort_values("Reviews", ascending=False).reset_index(drop=True)

print("\nSTEP 10 - COMPLAINT CATEGORIES")
print(group_df)

group_df.sort_values("Reviews").plot(
    kind="barh", x="Category", y="Reviews", color="#9b59b6", legend=False
)
plt.title("Critical Reviews by Complaint Category", fontsize=14)
plt.xlabel("Number of Critical Reviews")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

# INSIGHT: One review can mention more than one problem,
# so the percentages can add up to more than 100%.
# The top bar is the biggest area the company should fix first.

## Step 11: Insights Summary (Displayed as Markdown)

### 💡 Business Insight
This table connects **data analysis with business action**. The highest complaint category and keywords show which customer problem should be investigated first.


In [ ]:
# =================================================================
# STEP 11: SHOW OUR CALCULATED INSIGHTS AS A MARKDOWN TABLE
# =================================================================
# WHY: Tables look cleaner than printed text. This cell builds the
# markdown from the numbers we calculated, so it always matches the data.

insight_md = f"""## 📊 Calculated Insights

**Critical reviews (1-2 stars):** {len(df_critical):,} out of {len(df):,} total reviews ({critical_percent}%)

### 🔑 Top Complaint Keywords

| Rank | Keyword | Category | Critical Reviews Mentioning It |
|------|---------|----------|-------------------------------|
"""
for rank, row in enumerate(keyword_df.head(10).itertuples(), start=1):
    insight_md += f"| {rank} | {row.Keyword} | {row.Category} | {row.Reviews} |\n"

insight_md += """
### 🧩 Complaint Categories

| Category | Critical Reviews | % of Critical Reviews |
|----------|------------------|-----------------------|
"""
for _, row in group_df.iterrows():
    insight_md += f"| {row['Category']} | {row['Reviews']} | {row['% of Critical Reviews']}% |\n"

display(Markdown(insight_md))

# INSIGHT: This is the table to show in your submission.

## Step 12: Select the 3 Most Critical, Detailed Reviews

In [ ]:
# =================================================================
# STEP 12: PICK 3 REVIEWS FOR THE AI
# =================================================================
# WHY: We want reviews that are (a) the angriest = Rating 1, and
# (b) the most detailed = the longest. Long reviews give the AI
# more specific complaints to mention in the apology.
# We also remove repeated review texts so we do not pick the same one twice.

df_unique = df_critical.drop_duplicates(subset="Review Text")

top3 = (
    df_unique
    .sort_values(["Rating", "word_count"], ascending=[True, False])   # lowest rating first, then longest
    .head(3)
    .copy()
)

print("\nSTEP 12 - THE 3 SELECTED REVIEWS")
for number, (index, row) in enumerate(top3.iterrows(), start=1):
    print(f"\n--- Review {number} | Rating: {row['Rating']} | Words: {row['word_count']} | {row['Class Name']} ---")
    print("Title:", row["Title"])
    print("Text :", row["Review Text"])

# INSIGHT: These 3 reviews are 1-star and very detailed,
# so the apology emails can be specific instead of generic.

## Step 13: Connect to the Gen AI (Gemini)

In [ ]:
# =================================================================
# STEP 13: SET UP THE GEN AI CONNECTION
# =================================================================
# WHY: To write emails, we need to talk to a Gen AI model.
# We use Google Gemini. It needs a secret API key.
# SAFETY: The key is read from an environment variable, or typed in
# a hidden box. It is never saved inside the notebook.

MODEL_NAME = "gemini-3.8-flash"   # if you get a "model not found" error, check the current name at ai.google.dev/gemini-api/docs/models

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    api_key = getpass.getpass("Paste your Gemini API key (hidden): ")

try:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=api_key)
    print("\nSTEP 13 - GEN AI CLIENT READY")
    print("Model:", MODEL_NAME)
except ImportError:
    print("\nSTEP 13 - google-genai is not installed.")
    print("Run this once in a cell, then restart:  pip install google-genai")

# INSIGHT: If you see "CLIENT READY", the next steps can call the AI.

## Step 14: Write the Prompt

In [ ]:
# =================================================================
# STEP 14: PROMPT FOR THE CUSTOMER SUPPORT AGENT
# =================================================================
# WHY: The AI only writes a good email if we give clear instructions.
# SYSTEM_PROMPT = who the AI is and the rules it must follow.
# build_user_prompt = the specific review for each email.
# We tell it NOT to invent order numbers or discount codes,
# so the email never promises something the company did not offer.

SYSTEM_PROMPT = """You are a friendly and professional Customer Support Agent at an online women's clothing store.
Your job is to write a short apology email to a customer who left a very negative review.

Rules:
1. Start with a subject line in this format: "Subject: ..."
2. Greet the customer with "Dear Valued Customer," because we do not know their name.
3. Sincerely apologise and mention the SPECIFIC problems from their review (fit, fabric, quality, etc.).
4. Show real empathy. The customer should feel heard, not like they received a copy-paste reply.
5. Offer one clear next step: our support team can help with a return, exchange or refund.
6. Do NOT invent order numbers, discount codes, names, dates or company policies.
7. Keep the email body under 120 words.
8. End with "Warm regards," followed by "Customer Support Team"."""

def build_user_prompt(row):
    """Put one review's details into a prompt for the AI."""
    return f"""Customer review details:
- Star rating: {row['Rating']} out of 5
- Product type: {row['Class Name']} ({row['Department Name']})
- Review title: {row['Title']}
- Review text: {row['Review Text']}

Write the apology email now."""

def generate_apology_email(row):
    """Send one review to Gemini and return the email text."""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_user_prompt(row),
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT),
    )
    return response.text.strip()

print("\nSTEP 14 - PROMPT READY")
print("Example prompt for Review 1:\n")
print(build_user_prompt(top3.iloc[0]))

# INSIGHT: The prompt sends the rating, product type, title and the
# full review text, so the AI can personalise each email.

## Step 15: Generate the 3 Apology Emails

In [ ]:
# =================================================================
# STEP 15: GENERATE THE EMAILS
# =================================================================
# WHY: We loop through the 3 reviews, send each one to the AI,
# and save the answer. If one call fails (for example, a wrong key
# or no internet), we save the error message instead of crashing.
# The short pause between calls helps avoid free-tier rate limits.

emails = []

print("\nSTEP 15 - GENERATING EMAILS")
for number, (index, row) in enumerate(top3.iterrows(), start=1):
    print(f"Writing email {number} of {len(top3)}...")
    try:
        email_text = generate_apology_email(row)
    except Exception as error:
        email_text = f"ERROR - could not generate this email. Reason: {error}"
    emails.append(email_text)
    time.sleep(2)

top3["AI Email"] = emails
print("Done!")

# INSIGHT: If any email starts with "ERROR", check your API key,
# model name and internet connection, then run this cell again.

## Step 16: Show the Emails (Displayed as Markdown)

### 💡 Insight
The AI response should always be compared with the original review. GenAI is being used to **assist customer-support staff**, not replace human checking.


In [ ]:
# =================================================================
# STEP 16: DISPLAY AND SAVE THE EMAILS
# =================================================================
# WHY: We show each original review next to the AI's reply so the
# reader can check that the email really matches the complaint.
# We also save everything to a CSV file for the support team.

def make_markdown_safe(text):
    """Keep line breaks and stop '$' from being read as maths."""
    return text.replace("$", "\\$").replace("\n", "  \n")

display(Markdown("## ✉️ AI-Generated Apology Emails"))

for number, (index, row) in enumerate(top3.iterrows(), start=1):
    review_quote = make_markdown_safe(row["Review Text"])
    email_body = make_markdown_safe(row["AI Email"])
    display(Markdown(f"""### Email {number}  |  {row['Rating']}⭐  |  {row['Class Name']}

**Customer review:**

> {review_quote}

**AI-generated response:**

{email_body}

---"""))

top3[["Rating", "Title", "Class Name", "Review Text", "AI Email"]].to_csv(
    "generated_emails.csv", index=False
)
print("Emails saved to generated_emails.csv")

# INSIGHT: Each email should mention the specific problems from the
# review above it. If it feels generic, improve the rules in Step 14.

## Step 17: Final Insights & Conclusion

### 🎯 Final Takeaway
**Customer Reviews → Clean Data → Critical Reviews → Complaint Patterns → Business Insight → AI-Assisted Response**

This project turns customer feedback into an action-oriented customer-support workflow.


In [ ]:
# =================================================================
# STEP 17: FINAL STORY & CONCLUSION
# =================================================================
# WHY: A good data project must end with a simple summary
# that anyone can understand, not just people who read the code.
# All numbers below are calculated from the data, not typed by hand.

top_keyword = keyword_df.iloc[0]["Keyword"] if len(keyword_df) > 0 else "N/A"
top_category = group_df.iloc[0]["Category"] if len(group_df) > 0 else "N/A"
top_category_pct = group_df.iloc[0]["% of Critical Reviews"] if len(group_df) > 0 else 0
emails_ok = sum(1 for e in emails if not e.startswith("ERROR"))

print("\n" + "=" * 70)
print("FINAL INSIGHTS: CUSTOMER FEEDBACK AUTOMATION")
print("=" * 70)
print(f"""
1. DATA CLEANING:
   - Started with {rows_loaded:,} reviews.
   - Removed {rows_loaded - rows_after_missing:,} rows with no review text.
   - Removed {duplicate_count} duplicate rows.
   - Cleaned the text (lowercase, no symbols, no extra spaces).

2. RULE-BASED FILTERING (NO ML):
   - Rule used: Rating 1 or 2 = Critical.
   - Found {len(df_critical):,} critical reviews ({critical_percent}% of all reviews).

3. WHAT CUSTOMERS COMPLAIN ABOUT:
   - Most common complaint keyword: "{top_keyword}".
   - Biggest complaint category: {top_category} ({top_category_pct}% of critical reviews).

4. GEN AI OUTREACH:
   - Picked the 3 longest 1-star reviews.
   - Gemini drafted {emails_ok} of {len(emails)} personalised, empathetic emails.
   - Emails saved in generated_emails.csv.

NOTE: A human should read each AI email before sending it to a customer.
""")
print("PROJECT COMPLETE ✅")

## Step 18: Limitations & Future Work

### 🗣️ One-Line Viva Insight
**“We identify critical customer reviews using ratings, find the main complaint patterns with keywords, and use GenAI to create personalized support responses.”**


In [ ]:
# =================================================================
# STEP 18: LIMITATIONS & FUTURE WORK (SHORT VERSION)
# =================================================================
# WHY: Be honest about what this solution CAN'T do, and suggest improvements.

print("\n" + "=" * 60)
print("STEP 18: WHAT WAS THE PROBLEM? HOW TO FIX IT?")
print("=" * 60)

print("""
❌ PROBLEMS WE FACED:
1. Star rating is not perfect: Some 3-star reviews can be urgent,
   and some 1-star reviews are only about size.
2. Keyword list is hand-made: It can miss new or misspelled words
   (e.g. "reely bad") and cannot understand sarcasm.
3. AI can make mistakes: It may sound too generic or promise help
   the company cannot give.

✅ FUTURE IMPROVEMENTS:
1. Add an urgency score: Combine rating + strong words
   (e.g. "refund", "never again") to rank the worst reviews first.
2. Grow the keyword list: Use the top words from Step 8 to find
   new complaint words every week.
3. Human approval step: Let a support agent review each AI draft
   before it is sent.
""")

print("-" * 60)
print("CONCLUSION: Simple rules found the unhappy customers fast,")
print("and Gen AI turned them into ready-to-edit apology emails.")
print("-" * 60)